> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# 09_3 — LlamaのMAIN：MP両端保持 × DEの結果分析
09の保存済みLlama結果だけを使用する。**GPU・モデル取得・SVD・DE・PPL再評価は不要。CPUで「すべて実行」する。** Gemma、途中までのE1/E2、09_2の全層同時LRAは分析対象に含めない。元の実験フォルダには書き込まない。

- H：MP上端閾値より上の特異成分のみ保持。
- LH：下端閾値より下のLとHを保持。中央Bは0。
- top_same_rank_LH：LHと同じ個数を、大きい特異値から保持する対照。
- DEなしとありでは、各スペクトルの閾値に基づき独立にrankが決まる。**08_2の同rank比較とは異なる。**

各条件で元のモデルの1行列だけを変更する実験であり、全行列を同時に変更した結果ではない。保持成分数・理論上の因子化削減率をPPLと併記する。L=0の行列はHとLHが同一になり、下側追加効果を検証できないため区別する。

必要ライブラリ：numpy、pandas、matplotlib。ColabでもCPUで実行可能。Drive未接続時だけマウントを行う。APIキー不要。

In [ ]:
from pathlib import Path
import json, hashlib, platform, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

DATA_ROOT_OVERRIDE = None  # 通常変更不要。必要ならhashiguchi/dataを指定。
TAG = 'two-edge-lra-v1'
VERIFY_LOSSES = True  # MAINの保存損失・チェックサム・PPLをCPUで検証する。
HEATMAP_LIMIT = 10.0  # 色は改善率±10%で飽和。正確な数値はCSVに全件保存。
TIE_EPS_PCT = 0.01  # 改善率±0.01%を軽微な差と表示。検定ではない。
candidates=[Path('/content/drive/MyDrive/TUS/hashiguchi/data'),
 Path.home()/'Library/CloudStorage/GoogleDrive-ryoya.zushi1210@gmail.com/マイドライブ/TUS/hashiguchi/data']
if Path('/content').exists() and not candidates[0].exists():
    from google.colab import drive
    drive.mount('/content/drive')
DATA_ROOT=Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else next((p for p in candidates if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('DATA_ROOT_OVERRIDEにhashiguchi/dataを指定してください')
SOURCE=DATA_ROOT/'ibis2026_de_lra'/TAG
OUTPUT=DATA_ROOT/'ibis2026_de_lra'/(TAG+'-llama-main-analysis')
OUTPUT.mkdir(parents=True,exist_ok=True)
LABELS=['H','LH','top_same_rank_LH']
MODULES=['up_proj','down_proj','gate_proj','q_proj','k_proj','v_proj','o_proj']
COLORS={'H':'#4477AA','LH':'#EE6677','top_same_rank_LH':'#228833'}
plt.rcParams.update({'figure.dpi':120,'axes.grid':True,'grid.alpha':.2})
def digest(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def table(d,name): d.to_csv(OUTPUT/(name+'.csv'),index=False)
def figure(fig,name):
    fig.tight_layout();fig.savefig(OUTPUT/(name+'.png'),bbox_inches='tight');fig.savefig(OUTPUT/(name+'.pdf'),bbox_inches='tight')
    plt.show();plt.close(fig)
print('Read only:',SOURCE,'\nSave:',OUTPUT)


## 1. MAINの完了確認と保存結果の検証
28ブロック×7行列×DE有無×3保持法＝**1,176ラベル条件**を検証する。同一maskのラベルは同じPPL評価を共有するため、独立した評価が1,176回あったとは限らない。

欠損は`coverage.csv`に保存して停止する。未完了条件を除いた偏った主集計を作らない。保存設定・行列名・rank・maskを検証し、既定では損失ファイルのチェックサム、評価トークン数、PPLの再計算まで行う。途中のE1/E2を待つ必要はない。

In [ ]:
manifest=json.loads((SOURCE/'manifest.json').read_text());CONFIG_HASH=manifest['config_hash']
assert set(MODULES)==set(manifest['settings']['modules'])
cache={};receipt={'manifest_sha256':sha(SOURCE/'manifest.json'),'config_hash':CONFIG_HASH,'source_tag':TAG,
 'model':'llama','revision':manifest['settings']['revisions']['llama'],'verify_losses':VERIFY_LOSSES,
 'heatmap_limit':HEATMAP_LIMIT,'tie_eps_pct':TIE_EPS_PCT,'python':platform.python_version(),'files':{}}
def read_saved(path):
    if path in cache:return cache[path]
    summary=path/'summary.json'
    if not summary.exists():return None
    obj=json.loads(summary.read_text())
    assert obj['model']=='llama' and obj['config_hash']==CONFIG_HASH, f'Config/model mismatch: {path}'
    assert not obj.get('ppl_overflow',False) and obj['ppl'] is not None, f'PPL overflow: {path}'
    assert np.isfinite(obj['ppl']) and obj['ppl']>0
    receipt['files'][str(summary.relative_to(SOURCE))]=sha(summary)
    if VERIFY_LOSSES:
        loss_path=path/'losses.npz'
        assert sha(loss_path)==obj['loss_hash'],f'Loss checksum mismatch: {path}'
        with np.load(loss_path,allow_pickle=False) as z: loss=z['loss'];count=z['count']
        assert loss.ndim==count.ndim==1 and len(loss)==len(count)>0
        assert np.isfinite(loss).all() and (count==manifest['settings']['seq_len']-1).all()
        assert len(loss)==282, f'Llama evaluation block count changed: {path}'
        assert np.isclose(float(np.exp(loss.sum()/count.sum())),obj['ppl'],rtol=1e-8,atol=1e-10)
        receipt['files'][str(loss_path.relative_to(SOURCE))]=obj['loss_hash']
    cache[path]=obj
    return obj
base=read_saved(SOURCE/'llama/baseline');assert base is not None
BASELINE=base['ppl'];rows=[];coverage=[]
for block in range(28):
    for module in MODULES:
        family='mlp' if module in ['up_proj','down_proj','gate_proj'] else 'self_attn'
        name=f'model.layers.{block}.{family}.{module}'
        for de in [False,True]:
            directory=SOURCE/'llama/matrices'/digest(name)[:20]/str(int(de))
            p=directory/'plan.json'
            plan=json.loads(p.read_text()) if p.exists() else None
            if plan:
                assert plan['config_hash']==CONFIG_HASH and plan['name']==name and plan['de']==de
                receipt['files'][str(p.relative_to(SOURCE))]=sha(p)
                spec=plan['spectrum'];pdim=int(spec['p']);ndim=int(spec['n'])
                L=np.array(spec['L'],dtype=int);H=np.array(spec['H'],dtype=int);B=np.array(spec['B'],dtype=int)
                assert np.array_equal(np.sort(np.concatenate([L,B,H])),np.arange(pdim))
                expected={'H':np.unique(H).tolist(),'LH':np.union1d(L,H).tolist(),
                          'top_same_rank_LH':list(range(len(L)+len(H)))}
            for label in LABELS:
                found=[c for c in plan['conditions'] if c['phase']=='MAIN' and c['label']==label] if plan else []
                assert len(found)<=1
                c=found[0] if found else None;obj=None
                if c:
                    assert c['indices']==expected[label] and c['rank']==len(c['indices'])
                    mask=digest(c['indices'])[:24];obj=read_saved(directory/mask)
                    if obj:
                        assert obj['name']==name and obj['de']==de and obj['rank']==c['rank'] and obj['mask_id']==mask
                        assert np.isclose(obj['baseline_ppl'],BASELINE)
                        expected_reduction=max(0,1-c['rank']*(pdim+ndim)/(pdim*ndim))
                        assert np.isclose(obj['effective_reduction'],expected_reduction)
                        rows.append(dict(name=name,module=module,block=block,de=de,label=label,rank=c['rank'],
                         p=pdim,n=ndim,rank_fraction=c['rank']/pdim,n_low=len(L),n_high=len(H),
                         lower_fraction=len(L)/pdim,lower_status=spec['lower_status'],mask_id=mask,
                         ppl=obj['ppl'],baseline_ppl=BASELINE,ppl_ratio=obj['ppl']/BASELINE,
                         matrix_reduction=obj['effective_reduction'],model_reduction=obj['model_reduction'],
                         relative_frobenius_error=obj['relative_frobenius_error']))
                coverage.append(dict(name=name,module=module,block=block,de=de,label=label,
                     status='complete' if obj else ('missing_plan' if not plan else 'missing_result')))
    print(f'Validated block {block+1}/28',flush=True)
coverage=pd.DataFrame(coverage);table(coverage,'coverage')
display(coverage.groupby(['de','label','status']).size().rename('count').reset_index())
assert coverage.status.eq('complete').all(), 'MAIN未完了。coverage.csvを確認してください。実験の再実行はこの分析では行いません。'
results=pd.DataFrame(rows);assert len(results)==1176 and not results.duplicated(['name','de','label']).any()
receipt.update(conditions=len(results),unique_evaluations_excluding_baseline=len(cache)-1,baseline_ppl=BASELINE)
(OUTPUT/'input_audit.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2))
table(results,'main_results')
print('PASS: MAIN complete. Baseline PPL =',BASELINE,'Unique evaluations =',len(cache)-1)


## 2. 6条件の全体像：PPL・保持rank・削減率
PPLは小さいほど良い。PPL/baseline=1は無圧縮モデルと同じ値。保持rank比はrank/min(m,n)。削減率はrank(m+n)に基づく理論値を0以上に丸めたもの。実行はdense行列であり、実測の速度・メモリ削減を示さない。

行列ごとの中央値は、全行列を一度にLRAしたモデルのPPLではない。

In [ ]:
summary=results.groupby(['de','label']).agg(n=('ppl','size'),median_ppl=('ppl','median'),max_ppl=('ppl','max'),
 median_ppl_ratio=('ppl_ratio','median'),median_rank_fraction=('rank_fraction','median'),
 median_matrix_reduction=('matrix_reduction','median'),lower_present=('n_low',lambda s:int((s>0).sum()))).reset_index()
table(summary,'overall_summary');display(summary)
module_summary=results.groupby(['module','de','label']).agg(n=('ppl','size'),median_ppl=('ppl','median'),
 median_ppl_ratio=('ppl_ratio','median'),median_rank_fraction=('rank_fraction','median'),
 median_matrix_reduction=('matrix_reduction','median')).reset_index()
table(module_summary,'module_summary')
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,de in zip(axes,[False,True]):
    h=results[results.de==de]
    for label,g in h.groupby('label'):ax.scatter(100*g.matrix_reduction,g.ppl_ratio,label=label,c=COLORS[label],s=15,alpha=.6)
    ax.axhline(1,c='black',ls='--');ax.set(yscale='log',xlabel='Theoretical matrix reduction (%)',ylabel='PPL / baseline',title=f'Llama | DE={de}')
axes[-1].legend(fontsize=8);figure(fig,'ppl_vs_reduction')
fig,axes=plt.subplots(2,2,figsize=(14,8))
for j,de in enumerate([False,True]):
    for k,label in enumerate(LABELS):
        h=module_summary[(module_summary.de==de)&(module_summary.label==label)].set_index('module').loc[MODULES]
        x=np.arange(7)+(k-1)*.23
        axes[0,j].plot(x,h.median_ppl_ratio,linestyle='none',marker='o',label=label,color=COLORS[label])
        axes[1,j].bar(x,100*h.median_rank_fraction,.23,color=COLORS[label])
    for ax in axes[:,j]:ax.set_xticks(range(7),[m.replace('_proj','').upper() for m in MODULES])
    axes[0,j].set(yscale='log',ylabel='Median PPL / baseline',title=f'DE={de}');axes[0,j].axhline(1,c='black',ls='--')
    axes[1,j].set(ylabel='Median retained rank (%)')
axes[0,1].legend(fontsize=8);figure(fig,'module_ppl_and_rank')


## 3. 下側成分の追加と、同rank対照
**LH vs H**：下側成分を追加する比較なのでrankが増える。改善しても「下側が他の成分より重要」とは言えない。

**LH vs top_same_rank_LH**：保持rankを揃えた比較。正の改善率は、同数の上位保持よりLHのPPLが低いことを表す。Lがあるケースと、mask自体が異なるケースを別集計する。

改善率 = 100 × (1 − PPL_LH/PPL_対照)。勝敗は説明用の±TIE_EPS_PCTであり、統計的検定ではない。

In [ ]:
wide=results.pivot(index=['name','module','block','de'],columns='label',values=['ppl','rank','mask_id']).copy()
wide.columns=[f'{a}_{b}' for a,b in wide.columns];within=wide.reset_index()
within=within.merge(results[results.label=='LH'][['name','de','n_low','lower_fraction','lower_status']],on=['name','de'],validate='one_to_one')
for label in LABELS:
    within['ppl_'+label]=pd.to_numeric(within['ppl_'+label]);within['rank_'+label]=pd.to_numeric(within['rank_'+label])
assert (within.rank_LH==within.rank_top_same_rank_LH).all()
within['gain_add_lower']=100*(1-within.ppl_LH/within.ppl_H)
within['gain_same_rank']=100*(1-within.ppl_LH/within.ppl_top_same_rank_LH)
within['added_rank']=within.rank_LH-within.rank_H
within['distinct_same_rank_mask']=within.mask_id_LH!=within.mask_id_top_same_rank_LH
assert (within.added_rank==within.n_low).all()
assert np.allclose(within.loc[within.n_low==0,'gain_add_lower'],0)
table(within,'within_de_comparisons')
def aggregate_gain(g,col):
    x=g[col]
    return dict(n=len(g),improved=int((x>TIE_EPS_PCT).sum()),near_zero=int((x.abs()<=TIE_EPS_PCT).sum()),
                worsened=int((x < -TIE_EPS_PCT).sum()),median_gain=float(x.median()) if len(x) else np.nan)
rows=[]
for de,g in within.groupby('de'):
    for scope,h in [('all',g),('L_present',g[g.n_low>0]),('different_masks',g[g.distinct_same_rank_mask])]:
        for metric in ['gain_add_lower','gain_same_rank']:
            rows.append(dict(de=de,scope=scope,metric=metric,**aggregate_gain(h,metric)))
within_summary=pd.DataFrame(rows);table(within_summary,'within_de_summary');display(within_summary)
fig,axes=plt.subplots(2,2,figsize=(12,10))
for j,de in enumerate([False,True]):
    h=within[(within.de==de)&(within.n_low>0)]
    for i,(ref,title) in enumerate([('H','Add lower: rank increases'),('top_same_rank_LH','Same-rank comparison')]):
        ax=axes[i,j];ax.scatter(h['ppl_'+ref],h.ppl_LH,c=h.lower_fraction,cmap='viridis',s=25)
        lim=[min(h['ppl_'+ref].min(),h.ppl_LH.min())*.9,max(h['ppl_'+ref].max(),h.ppl_LH.max())*1.1] if len(h) else [1,10]
        ax.plot(lim,lim,c='black',ls='--');ax.set(xscale='log',yscale='log',xlim=lim,ylim=lim,
          xlabel=f'PPL: {ref}',ylabel='PPL: LH',title=f'{title} | DE={de} | L>0, n={len(h)}')
figure(fig,'lower_retention_paired')


## 4. DEの有無による差とブロック別ヒートマップ
DE改善率 = 100 × (1 − PPL_DE/PPL_raw)。**DE前後でrankが変わるため、DEの同rankでの効果とは区別する。** 正の色は比較対象よりPPLが低いことを表す。

色は±HEATMAP_LIMITで飽和し、それを超えた件数をタイトルに表示する。灰色は下側成分がなく、下側追加効果の比較対象外。全数値はCSVに保存する。

In [ ]:
raw=results[~results.de].drop(columns='de');de=results[results.de].drop(columns='de')
between=raw.merge(de,on=['name','module','block','label'],suffixes=('_raw','_de'),validate='one_to_one')
between['gain_de']=100*(1-between.ppl_de/between.ppl_raw)
between['delta_rank']=between.rank_de-between.rank_raw
between['delta_reduction']=between.matrix_reduction_de-between.matrix_reduction_raw
table(between,'between_de_comparisons')
rows=[]
for label,g in between.groupby('label'):
    rows.append(dict(label=label,**aggregate_gain(g,'gain_de'),median_delta_rank=g.delta_rank.median(),
      rank_equal=int((g.delta_rank==0).sum()),rank_less_DE=int((g.delta_rank<0).sum()),rank_more_DE=int((g.delta_rank>0).sum())))
between_summary=pd.DataFrame(rows);table(between_summary,'between_de_summary');display(between_summary)
def heatmap(g,col,title,name,mask_lower=False):
    h=g.copy()
    if mask_lower:h.loc[h.n_low==0,col]=np.nan
    z=h.pivot(index='module',columns='block',values=col).reindex(index=MODULES,columns=range(28))
    clipped=int((z.abs()>HEATMAP_LIMIT).sum().sum());cmap=plt.get_cmap('RdBu').copy();cmap.set_bad('#dddddd')
    fig,ax=plt.subplots(figsize=(15,4));im=ax.imshow(z.to_numpy(float),aspect='auto',cmap=cmap,
         norm=TwoSlopeNorm(vmin=-HEATMAP_LIMIT,vcenter=0,vmax=HEATMAP_LIMIT))
    ax.set_yticks(range(7),[m.replace('_proj','').upper() for m in MODULES]);ax.set_xticks(range(28));ax.set_xlabel('Block index (0-based)')
    ax.set_title(f'{title} | saturated cells: {clipped}; gray: L=0 / unavailable')
    ax.grid(False);fig.colorbar(im,ax=ax,label='PPL improvement (%)',extend='both');figure(fig,name)
for de,g in within.groupby('de'):
    for col in ['gain_add_lower','gain_same_rank']:heatmap(g,col,f'{col}, DE={de}',f'{col}_DE{int(de)}',True)
for label,g in between.groupby('label'):heatmap(g,'gain_de',f'DE improvement: {label}',f'DE_gain_{label}')
fig,ax=plt.subplots(figsize=(8,5))
for label,g in between.groupby('label'):ax.scatter(100*g.delta_reduction,g.gain_de,label=label,color=COLORS[label],alpha=.6,s=18)
ax.axhline(0,c='black',lw=.7);ax.axvline(0,c='black',lw=.7)
ax.set(xlabel='DE - raw matrix reduction (percentage points)',ylabel='DE PPL improvement (%)',title='DE comparison: reduction also changes');ax.legend();figure(fig,'de_gain_vs_reduction_change')


## 5. ゼミ用の注目例と要約
全体を確認した後の例示であり、独立した確認実験ではない。極端な行列だけでなく全体集計と併せて読む。H→LHの改善はrank増加を含み、下側成分固有の有効性は同rank対照で確認する。

ヒートマップに現れる改善・悪化のブロックを特定し、必要なら既存スペクトル可視化と照合する。このノートブックでは新しいモデル実験を起動しない。

In [ ]:
example_parts=[]
for de,g in within[within.n_low>0].groupby('de'):
    for metric in ['gain_add_lower','gain_same_rank']:
        for direction,h in [('best',g.nlargest(5,metric)),('worst',g.nsmallest(5,metric))]:
            h=h.copy();h['selection_metric']=metric;h['selection']=direction;example_parts.append(h)
examples=pd.concat(example_parts,ignore_index=True) if example_parts else pd.DataFrame()
table(examples,'selected_examples')
cols=['name','de','n_low','rank_H','rank_LH','ppl_H','ppl_LH','ppl_top_same_rank_LH','gain_add_lower','gain_same_rank','selection_metric','selection']
if len(examples):display(examples[cols])
lines=['# Llama MAIN：保存結果の集計','',f'- Baseline PPL: {BASELINE:.6f}',
 f'- 完了ラベル条件: {len(results)} / 1176',f'- 独立maskの保存評価数（baseline除く）: {len(cache)-1}',
 '- 単独行列LRA。Gemma・E1/E2・全層同時LRAを含まない。',
 '- 正の改善率は対照よりPPLが低い。無圧縮モデルより良いとは限らない。',
 '- H対LHおよびDE有無ではrank差を伴う。同rank対照はLH対top_same_rank_LH。','']
for row in within_summary.itertuples():
    if row.scope=='L_present':lines.append(f'- DE={row.de}, L>0, {row.metric}: n={row.n}, median={row.median_gain:.3f}%, improved/near-zero/worsened={row.improved}/{row.near_zero}/{row.worsened}')
for row in between_summary.itertuples():lines.append(f'- {row.label}, DE改善率中央値={row.median_gain:.3f}%, 同rank={row.rank_equal}/{row.n}')
lines+=['','改善/軽微/悪化は±'+str(TIE_EPS_PCT)+'%による記述分類であり、有意差検定ではない。',
 '因子化削減率は理論値。推論速度や実メモリの測定結果ではない。','図の色飽和に隠れる極端値はCSVと注目例表で確認する。']
(OUTPUT/'summary.md').write_text('\n'.join(lines),encoding='utf-8')
print('\n'.join(lines));print('\n保存先:',OUTPUT)
